In [1]:
!pip install xgboost

In [2]:
# ============================================
# Colab: Restaurant Rating Prediction Model
# Using restaurant.json
# ============================================

# !pip install xgboost==2.1.1 --quiet

import pandas as pd
import numpy as np
import json

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from xgboost import XGBRegressor
from scipy.stats import randint, uniform

# ===========================
# 1. Load data
# ===========================

# ---- restaurants from JSON ----
# Assumes restaurant.json is a list of restaurant objects like the one you showed.
restaurants = pd.read_json("/content/restaurant.json", orient="records")

# ---- CSVs ----
reviews = pd.read_csv("/content/reviews.csv")
user_data = pd.read_csv("/content/user_data.csv")
dining_trends = pd.read_csv("/content/dining_trends.csv")

print("restaurants.head():")
display(restaurants.head())
print("reviews.head():")
display(reviews.head())
print("user_data.head():")
display(user_data.head())
print("dining_trends.head():")
display(dining_trends.head())

# ===========================
# 2. Basic cleaning / joins
# ===========================

# restaurant.json has "id" → treat as restaurant_id
if "restaurant_id" not in restaurants.columns:
    if "id" in restaurants.columns:
        restaurants = restaurants.rename(columns={"id": "restaurant_id"})
    else:
        raise ValueError("restaurant.json must contain 'id' or 'restaurant_id'.")

# ---- Time-series features from dining_trends ----
if "date" in dining_trends.columns:
    dining_trends["date"] = pd.to_datetime(
        dining_trends["date"], dayfirst=True, errors="coerce"
    )

trend_numeric_cols = ["popularity_score", "avg_price", "booking_lead_time_days"]
trend_numeric_cols = [c for c in trend_numeric_cols if c in dining_trends.columns]

trend_agg = dining_trends.groupby("cuisine_type")[trend_numeric_cols].mean().reset_index()
trend_agg = trend_agg.rename(columns={"cuisine_type": "cuisine"})

# ---- Merge tables ----
df = reviews.merge(
    restaurants,
    on="restaurant_id",
    how="left",
    suffixes=("", "_rest")
)

df = df.merge(
    user_data,
    on="user_id",
    how="left",
    suffixes=("", "_user")
)

if "cuisine" in df.columns:
    df = df.merge(trend_agg, on="cuisine", how="left", suffixes=("", "_trend"))

print("\nMerged training dataframe:")
display(df.head())

df = df.dropna(subset=["rating"])  # rating in reviews.csv

# ===========================
# 3. Feature definitions
# ===========================

y = df["rating"].astype(float)
text_col = "review_text"

numeric_cols = []

# user_data numeric
for col in ["avg_rating_total_reviews_written", "age"]:
    if col in df.columns:
        numeric_cols.append(col)

# restaurant-level numeric from JSON
for col in ["rating_rest", "review_count"]:
    if col in df.columns:
        numeric_cols.append(col)

# trends numeric
for col in trend_numeric_cols:
    if col in df.columns:
        numeric_cols.append(col)

# price_range → numeric
def parse_price_range(s):
    if not isinstance(s, str):
        return np.nan
    nums = "".join(ch if ch.isdigit() or ch == " " else " " for ch in s)
    parts = [p for p in nums.split() if p.isdigit()]
    if len(parts) == 0:
        return np.nan
    vals = list(map(float, parts))
    return np.mean(vals)

if "price_range" in df.columns:
    df["avg_price_range"] = df["price_range"].apply(parse_price_range)
    numeric_cols.append("avg_price_range")

categorical_cols = []
for col in [
    "cuisine",           # from restaurant.json
    "location",          # from restaurant.json
    "home_location",     # from user_data.csv
    "dining_frequency",  # from user_data.csv
    "favorite_cuisines",
    "preferred_",
    "dietary_res"
]:
    if col in df.columns:
        categorical_cols.append(col)

print("\nNumeric columns used:", numeric_cols)
print("Categorical columns used:", categorical_cols)
print("Text column:", text_col if text_col in df.columns else "NOT FOUND")

# ===========================
# 4. Preprocessing pipeline
# ===========================

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

text_transformer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    stop_words="english"
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_cols),
        ("cat", categorical_transformer, categorical_cols),
        ("txt", text_transformer, text_col)
    ],
    remainder="drop"
)




restaurants.head():


,id,name,cuisine,location,price_range,description,amenities,attributes,opening_hours,coordinates,rating,review_count
0,1,Golden Dhow Cafe,Chinese,Al Barsha,AED 150 - 200,"This intimate, exclusive retreat using sleek m...",Outdoor Seating,"Vegetarian-Focused, Casual Dining",10:00 - 23:00,"25.374629, 55.408877",3.30,574
1,2,Classic Emirati House,Emirati,Al Barsha,AED 150 - 200,"A sophisticated, dual-level dining space featu...","Live Music, Delivery Available, Free WiFi, Out...","Best Views, Late-Night",9:00 - 21:00,"25.231947, 55.842095",3.12,793
2,3,Desert Palace,French,Downtown Dubai,AED 50 - 100,A beautifully designed space with floor-to-cei...,Free WiFi,"Late-Night, Vegetarian-Focused, Best Views",11:00 - 22:00,"24.869259, 55.497432",3.15,246
3,4,The Original French House,French,Sharjah,AED 200 - 300+,"A sophisticated, dual-level dining space featu...","Live Music, Free WiFi, Valet Parking, Wheelcha...",Late-Night,11:00 - 22:00,"25.361727, 55.594466",4.60,545
4,5,The Chinese Spot,Chinese,Abu Dhabi,AED 50 - 100,"The bright, airy dining room featuring hand-pa...","Wheelchair Accessible, Valet Parking, Reservat...","Best Views, Business Lunch",10:00 - 23:00,"25.458541, 55.810312",4.12,457


reviews.head():


,review_id,restaurant_id,user_id,rating,review_text,date,helpful_count
0,1,5,10001,2,The server was clearly untrained and had no kn...,2024-07-26,13
1,2,1,10002,3,"I appreciate the effort, but the music was far...",2024-02-05,28
2,3,3,10003,3,"The ambiance was great, but the service team s...",2025-03-24,14
3,4,27,10004,3,"It was a busy night, and you could feel the pr...",2025-04-06,20
4,5,28,10005,4,They nailed the ambiance. It felt like a true ...,2023-11-18,4


user_data.head():


...

,date,cuisine_type,popularity_score,avg_price,season,day_type,is_holiday,weather_impact_category,booking_lead_time_days
0,2023-11-19,Italian,56.51,139.02,Winter/Peak,Weekday,False,Neutral,0
1,2023-11-19,Chinese,69.77,176.86,Winter/Peak,Weekday,False,Neutral,1
2,2023-11-19,Indian,80.87,113.12,Winter/Peak,Weekday,False,Neutral,6
3,2023-11-19,Mexican,61.96,185.22,Winter/Peak,Weekday,False,Neutral,0
4,2023-11-19,French,68.17,257.61,Winter/Peak,Weekday,False,Neutral,1



Merged training dataframe:


/tmp/ipython-input-1542884600.py:58: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  dining_trends["date"] = pd.to_datetime(


,review_id,restaurant_id,user_id,rating,review_text,date,helpful_count,name,cuisine,location,...,home_location,dining_frequency,favorite_cuisines,preferred_price_range,dietary_restrictions,avg_rating_given,total_reviews_written,popularity_score,avg_price,booking_lead_time_days
0,1,5,10001,2,The server was clearly untrained and had no kn...,2024-07-26,13,The Chinese Spot,Chinese,Abu Dhabi,...,Sharjah,Daily,"Mexican, Emirati, Mediterranean",Medium,Halal,3.30,134,65.830862,177.903639,1.783858
1,2,1,10002,3,"I appreciate the effort, but the music was far...",2024-02-05,28,Golden Dhow Cafe,Chinese,Al Barsha,...,Dubai Marina,Bi-Weekly,"French, Mediterranean, Italian",Medium,NaN,3.57,98,65.830862,177.903639,1.783858
2,3,3,10003,3,"The ambiance was great, but the service team s...",2025-03-24,14,Desert Palace,French,Downtown Dubai,...,Jumeirah Lakes Towers (JLT),Weekly,"French, Chinese",Medium,Gluten-Free,3.46,7,71.803858,306.788345,2.761970
3,4,27,10004,3,"It was a busy night, and you could feel the pr...",2025-04-06,20,Central Thai Cafe,Thai,Downtown Dubai,...,Palm Jumeirah,Bi-Weekly,"Emirati, Chinese",High,Vegan,4.04,23,65.844446,179.672613,1.753762
4,5,28,10005,4,They nailed the ambiance. It felt like a true ...,2023-11-18,4,Coast Iranian Spot,Iranian,Ajman,...,Abu Dhabi,Bi-Weekly,"Emirati, French",Medium,Gluten-Free,4.32,121,66.332873,181.768741,1.826265



Numeric columns used: ['age', 'rating_rest', 'review_count', 'popularity_score', 'avg_price', 'booking_lead_time_days', 'avg_price_range']
Categorical columns used: ['cuisine', 'location', 'home_location', 'dining_frequency', 'favorite_cuisines']
Text column: review_text


In [4]:
# ===========================
# 5. Model + Pipeline
# ===========================

model = XGBRegressor(
    objective="reg:squarederror",
    tree_method="hist",
    eval_metric="rmse",
    random_state=42,
    n_jobs=-1
)

pipeline = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", model)
])

# ===========================
# 6. Train / validation split
# ===========================

X = df[numeric_cols + categorical_cols + [text_col]]
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("\nTrain size:", X_train.shape, "Valid size:", X_valid.shape)

# ===========================
# 7. Hyperparameter tuning
# ===========================

param_distributions = {
    "model__n_estimators": randint(200, 600),
    "model__max_depth": randint(3, 8),
    "model__learning_rate": uniform(0.01, 0.2),
    "model__subsample": uniform(0.6, 0.4),
    "model__colsample_bytree": uniform(0.6, 0.4),
    "model__min_child_weight": randint(1, 8),
}

search = RandomizedSearchCV(
    pipeline,
    param_distributions=param_distributions,
    n_iter=20,
    scoring="neg_root_mean_squared_error",
    cv=3,
    verbose=1,
    n_jobs=-1,
    random_state=42
)

search.fit(X_train, y_train)

print("\nBest parameters from RandomizedSearchCV:")
print(search.best_params_)

# ===========================
# 8. Evaluation
# ===========================
import joblib


best_model = search.best_estimator_
joblib.dump(best_model, "rating_model.joblib")
print("Model saved as rating_model.joblib")

y_pred = best_model.predict(X_valid)

rmse = np.sqrt(mean_squared_error(y_valid, y_pred))
mae = mean_absolute_error(y_valid, y_pred)
r2 = r2_score(y_valid, y_pred)

print("\nValidation metrics:")
print(f"RMSE: {rmse:.4f}")
print(f"MAE : {mae:.4f}")
print(f"R²  : {r2:.4f}")

# ===========================
# 9. Structured feature importances
# ===========================

structured_features = numeric_cols + categorical_cols

struct_preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_cols),
        ("cat", categorical_transformer, categorical_cols),
    ],
    remainder="drop"
)

struct_model = XGBRegressor(
    objective="reg:squarederror",
    tree_method="hist",
    eval_metric="rmse",
    random_state=42,
    n_jobs=-1
)

struct_pipeline = Pipeline(steps=[
    ("preprocess", struct_preprocessor),
    ("model", struct_model)
])

X_train_s = df[structured_features].iloc[X_train.index]
X_valid_s = df[structured_features].iloc[X_valid.index]

struct_pipeline.fit(X_train_s, y_train)

ohe = struct_pipeline.named_steps["preprocess"].named_transformers_["cat"].named_steps["onehot"]
cat_feature_names = list(ohe.get_feature_names_out(categorical_cols)) if categorical_cols else []
struct_feature_names = numeric_cols + cat_feature_names

importances = struct_pipeline.named_steps["model"].feature_importances_
indices = np.argsort(importances)[::-1][:20]

print("\nTop structured features by importance (XGBoost):")
for idx in indices:
    if idx < len(struct_feature_names):
        print(f"{struct_feature_names[idx]}: {importances[idx]:.4f}")




Train size: (800, 13) Valid size: (200, 13)
Fitting 3 folds for each of 20 candidates, totalling 60 fits

Best parameters from RandomizedSearchCV:
{'model__colsample_bytree': np.float64(0.662397808134481), 'model__learning_rate': np.float64(0.021616722433639893), 'model__max_depth': 7, 'model__min_child_weight': 4, 'model__n_estimators': 559, 'model__subsample': np.float64(0.8832290311184181)}

Validation metrics:
RMSE: 0.5135
MAE : 0.4016
R²  : 0.8668

Top structured features by importance (XGBoost):
cuisine_Chinese: 0.0419
location_Ajman: 0.0207
location_Dubai Marina: 0.0188
favorite_cuisines_Emirati, Indian: 0.0175
dining_frequency_Weekly: 0.0172
dining_frequency_Rarely: 0.0149
favorite_cuisines_Thai: 0.0139
cuisine_Mexican: 0.0126
favorite_cuisines_Indian, Emirati: 0.0124
favorite_cuisines_Iranian: 0.0119
favorite_cuisines_Seafood, Thai: 0.0115
booking_lead_time_days: 0.0113
favorite_cuisines_Indian: 0.0111
home_location_Business Bay: 0.0110
favorite_cuisines_French: 0.0108
favori

In [ ]:
# ===========================
# 10. Notes (interpretability, drift, retraining)
# ===========================

notes = """
================ Documentation Notes ================

1. Model Interpretability
   - XGBoost provides feature importances for structured features.
   - For deeper insight (including text TF-IDF), use SHAP to see
     per-example contributions.

2. Handling Drift
   - Monitor distributions of inputs (cuisine, price_range, locations, user
     segments, popularity_score) vs training data.
   - Track rolling RMSE/MAE and trigger alerts when they degrade.
   - If big changes in cuisine mix / locations / user base, consider retraining.

3. Retraining Strategy
   - Retrain weekly or monthly using latest reviews + trends.
   - Rebuild TF-IDF vocabulary to capture new terms in review_text.
   - Keep model versions (champion/challenger) and A/B test online.

4. Production Monitoring
   - Monitor latency, error rate of prediction API.
   - Monitor alignment between predicted ratings and later actual user ratings.
   - Build dashboards (e.g. Prometheus/Grafana, cloud monitoring) to visualize
     metrics and drift.
"""
print(notes)

In [5]:
import joblib
best_model = search.best_estimator_
joblib.dump(best_model, "rating_model.joblib")
print("Model saved as rating_model.joblib")

Model saved as rating_model.joblib
